# In-the-Wild & additional datasets
Loads datasets via HuggingFace streaming — no local storage needed.

In [6]:
import os, sys
notebook_dir = os.getcwd()
if notebook_dir not in sys.path:
    sys.path.insert(0, notebook_dir)

def load_hf_token(path="secret.txt"):
    if os.path.exists(path):
        with open(path) as f:
            return f.read().strip()
    return None

HF_TOKEN = load_hf_token()
print("HF token:", "✅" if HF_TOKEN else "❌ No token — some datasets may be gated")

HF token: ✅


## 1. In-the-Wild (`mueller91/In-The-Wild`)
58 celebrities/politicians, real + synthetic speech. The standard OOD benchmark for deepfake detection generalization.

In [4]:
from datasets import load_dataset, Audio

ITW_CACHE = "./data/in_the_wild"

ds_itw = load_dataset(
    "mueller91/In-The-Wild",
    cache_dir=ITW_CACHE,
    token=HF_TOKEN if HF_TOKEN else None,
)

print("Splits:", ds_itw)
print("Columns:", ds_itw[list(ds_itw.keys())[0]].column_names)
print("First sample keys:", list(ds_itw[list(ds_itw.keys())[0]][0].keys()))

Splits: IterableDatasetDict({
    train: IterableDataset({
        features: ['audio'],
        n_shards: 1
    })
})
Columns: ['audio']


NotImplementedError: Subclasses of Dataset should implement __getitem__.

In [ ]:
# Inspect label distribution
from collections import Counter

split = list(ds_itw.keys())[0]
ds_itw_split = ds_itw[split]

# Cast to 16kHz
ds_itw_split = ds_itw_split.cast_column("audio", Audio(sampling_rate=16000))

# Find label column
sample = ds_itw_split[0]
print("Sample keys:", list(sample.keys()))

label_col = next((k for k in sample.keys() if "label" in k.lower() or k.lower() in ("key","speaker","type","class")), None)
print(f"Label column: {label_col}")

if label_col:
    labels = [ds_itw_split[i][label_col] for i in range(min(200, len(ds_itw_split)))]
    print("Label sample (first 200):", Counter(labels))
print(f"Total samples: {len(ds_itw_split)}")

In [ ]:
# Wrap with  existing loader utilities for use in training/eval
import torch
from loader import _label_to_int, _crop_policy, TARGET_SR

class ITWDataset(torch.utils.data.Dataset):
    """
    In-the-Wild dataset wrapper compatible with your existing training pipeline.
    Labels: bonafide=0, spoof=1
    """
    def __init__(self, hf_split, label_col, mode="eval"):
        self.ds        = hf_split
        self.label_col = label_col
        self.mode      = mode

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        ex  = self.ds[idx]
        wav = torch.tensor(ex["audio"]["array"], dtype=torch.float32).unsqueeze(0)
        wav = _crop_policy(wav, self.mode)
        y   = _label_to_int(ex[self.label_key])
        return {"audio": wav, "label": torch.tensor(y).long(), "sample_rate": TARGET_SR}

# NOTE: update label_col after inspecting the output above
ITW_LABEL_COL = label_col  # auto-detected above

itw_dataset = ITWDataset(ds_itw_split, ITW_LABEL_COL, mode="eval")
print(f"ITW dataset ready: {len(itw_dataset)} samples")

## 2. ASVspoof 2021 LA
Check what's available on HuggingFace — some mirrors exist but may be gated.

In [7]:
# Search for ASVspoof 2021 on HF
# The most commonly mirrored version:
ASV2021_OPTIONS = [
    "audioshake/asvspoof-2021-df",
    "lil-skies/asvspoof2021",
    "chkla/ASVspoof2021",
]

from datasets import load_dataset_builder

for name in ASV2021_OPTIONS:
    try:
        builder = load_dataset_builder(name, token=HF_TOKEN if HF_TOKEN else None)
        print(f"✅ {name} — accessible")
        print(f"   Info: {builder.info.description[:100] if builder.info.description else 'no description'}...")
    except Exception as e:
        print(f"❌ {name} — {str(e)[:80]}")

❌ audioshake/asvspoof-2021-df — Dataset 'audioshake/asvspoof-2021-df' doesn't exist on the Hub or cannot be acce
❌ lil-skies/asvspoof2021 — Dataset 'lil-skies/asvspoof2021' doesn't exist on the Hub or cannot be accessed
❌ chkla/ASVspoof2021 — Dataset 'chkla/ASVspoof2021' doesn't exist on the Hub or cannot be accessed


In [ ]:
# Load whichever one works — update DATASET_ID after running the cell above
ASV2021_ID    = "audioshake/asvspoof-2021-df"   # ← update if needed
ASV2021_CACHE = "./data/asvspoof2021"

try:
    ds_2021 = load_dataset(
        ASV2021_ID,
        cache_dir=ASV2021_CACHE,
        token=HF_TOKEN if HF_TOKEN else None,
    )
    ds_2021 = ds_2021[list(ds_2021.keys())[0]].cast_column("audio", Audio(sampling_rate=16000))
    print(f"✅ ASVspoof 2021 loaded: {len(ds_2021)} samples")
    print("Columns:", ds_2021.column_names)
    print("Sample:", {k: v for k, v in ds_2021[0].items() if k != "audio"})
except Exception as e:
    print(f"❌ Could not load: {e}")
    print("You may need to request access at https://huggingface.co/datasets/" + ASV2021_ID)

## 3. CodecFake (`rogertseng/CodecFake`)
Deepfake audio from codec-based TTS systems (VALL-E, EnCodec etc.) — tests robustness to modern neural codec spoofing.

In [ ]:
CODECFAKE_CACHE = "./data/codecfake"

try:
    ds_codec = load_dataset(
        "rogertseng/CodecFake",
        cache_dir=CODECFAKE_CACHE,
        token=HF_TOKEN if HF_TOKEN else None,
    )
    split = list(ds_codec.keys())[0]
    ds_codec_split = ds_codec[split].cast_column("audio", Audio(sampling_rate=16000))
    print(f"✅ CodecFake loaded: {len(ds_codec_split)} samples")
    print("Columns:", ds_codec_split.column_names)
    sample = ds_codec_split[0]
    label_col_cf = next((k for k in sample.keys() if "label" in k.lower() or k.lower() in ("key","type","class")), None)
    print(f"Label column: {label_col_cf}")
    if label_col_cf:
        labels_cf = [ds_codec_split[i][label_col_cf] for i in range(min(100, len(ds_codec_split)))]
        print("Label sample:", Counter(labels_cf))
except Exception as e:
    print(f"❌ Could not load CodecFake: {e}")

Resolving data files:   0%|          | 0/161 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/161 [00:00<?, ?it/s]

## 4. Evaluate your model on In-the-Wild
Drop-in evaluation against your existing checkpoint — no retraining needed.

In [ ]:
from argparse import Namespace
from phoneme_GAT.modules import Phoneme_GAT_lit
from pytorch_lightning import Trainer
from pytorch_lightning.loggers import WandbLogger
from torch.utils.data import DataLoader
from callbacks_rational import (
    BinaryACC_Callback, BinaryAUC_Callback, EER_Callback,
    TPR_Callback, TNR_Callback, FPR_Callback, FNR_Callback,
)
import wandb

cfg = Namespace(
    PhonemeGAT=Namespace(
        backbone="wavlm",
        use_raw=False,
        use_GAT=True,
        n_edges=10,
        use_aug=True,
        use_pool=True,
        use_clip=True,
    )
)

CKPT = "robust_goat.ckpt"   # ← swap to mixed_goat.ckpt or goat.ckpt to compare

model = Phoneme_GAT_lit.load_from_checkpoint(CKPT, cfg=cfg)
model = model.cuda()
model.eval()
torch.set_float32_matmul_precision("medium")
print(f"Loaded {CKPT}")

In [ ]:
itw_loader = DataLoader(
    itw_dataset,
    batch_size=20,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

if wandb.run is not None:
    wandb.finish()

itw_logger = WandbLogger(
    project="DeepfakeDetectionRenewed",
    entity="krishrawat0222-f",
    name=f"eval_in_the_wild_{CKPT.replace('.ckpt','')}",
    log_model=False,
    tags=["eval", "in-the-wild", "ood"],
)
itw_logger.experiment.config.update({
    "eval_dataset": "mueller91/In-The-Wild",
    "model_ckpt":   CKPT,
}, allow_val_change=True)

trainer = Trainer(
    accelerator="gpu",
    devices=1,
    logger=itw_logger,
    callbacks=[
        BinaryACC_Callback(batch_key="label", output_key="logit"),
        BinaryAUC_Callback(batch_key="label", output_key="logit"),
        EER_Callback(batch_key="label", output_key="logit"),
        TPR_Callback(batch_key="label", output_key="logit"),
        TNR_Callback(batch_key="label", output_key="logit"),
        FPR_Callback(batch_key="label", output_key="logit"),
        FNR_Callback(batch_key="label", output_key="logit"),
    ],
)

results = trainer.validate(model=model, dataloaders=itw_loader)
print("\nIn-the-Wild results:", results)

if wandb.run is not None:
    wandb.finish()